In [0]:
dbutils.widgets.text("batch_id" , "1" ,"Batch Id(1,2,or 3)")

In [0]:
batch_id = dbutils.widgets.get("batch_id")

In [0]:
from datetime import datetime

team_name  = "team_lemma"
silver_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
catalog  = f"charles_schwab_retailbrokerage_dev_{team_name}"
gold_db = f"charles_schwab_retailbrokerage_dev_{team_name}.gold"

spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:
## Extract the carried run_id from the bronze_table

try:
    run_info_now = spark.sql(f"""
                               select _run_id , _batch FROM {silver_db}.dailymarket 
                               where _batch = '{batch_id}'
                               LIMIT 1 
                            """).first()
    
    carried_run_id = run_info_now[0] if run_info_now[0] else "Unknown"
    carried_batch = run_info_now[1] if run_info_now[1] else batch_id
except Exception:
    carried_run_id = "Unknown"
    carried_batch = batch_id


run_id = carried_run_id
print(f"silver  : {silver_db}")
print(f"gold : {gold_db}")


In [0]:
from pyspark.sql.functions import (
    col, lit, trim, current_timestamp,
    date_format, when,
    min as spark_min,
    max as spark_max,
    row_number , lead , expr , unix_date , 
)
from pyspark.sql.window import Window
from pyspark.sql import Row
from pyspark.sql.types import *


In [0]:
df_silver = spark.table(f"{silver_db}.markethistory")

df_dim_sec = spark.table(f"{gold_db}.dim_security")
df_gold_fin = spark.table(f"{gold_db}.financial")


silver_count = df_silver.count()
print(f"Silver total rows : {silver_count:,}")



In [0]:
dm_with_sec = df_silver.join(
    df_dim_sec,
    (df_silver.dm_s_symb == df_dim_sec.symbol) &
    (df_silver.dm_date >= df_dim_sec.effectivedate) &
    (df_silver.dm_date < df_dim_sec.enddate),
    "left"
)

fin_gold_optimized = (
    df_gold_fin
    .withColumnRenamed("sk_companyid", "fin_sk_companyid")
)

w_fin = Window.partitionBy("fin_sk_companyid").orderBy("fi_qtr_start_date")

fin_gold_optimized = (
    fin_gold_optimized
    .withColumn("fi_qtr_end_date", lead("fi_qtr_start_date").over(w_fin))
    .withColumn("fi_qtr_end_date", when(col("fi_qtr_end_date").isNull(), expr("CAST('9999-12-31' AS DATE)")).otherwise(col("fi_qtr_end_date")))
)

dm_fin_join = dm_with_sec.join(
    fin_gold_optimized,
    (dm_with_sec.sk_companyid == fin_gold_optimized.fin_sk_companyid) &
    (dm_with_sec.dm_date >= fin_gold_optimized.fi_qtr_start_date) &
    (dm_with_sec.dm_date < fin_gold_optimized.fi_qtr_end_date),
    "left"
)

w_52 = Window.partitionBy("sk_securityid").orderBy(col("dm_date").cast("timestamp").cast("long")).rangeBetween(-(364 * 24 * 60 * 60), 0)

df_gold_market = (
    dm_fin_join
    .withColumn("sk_dateid", date_format("dm_date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("peratio", expr("try_divide(dm_close, fi_basic_eps)").cast(DecimalType(10,2)))
    .withColumn("yield", expr("try_divide(dividend, dm_close)").cast(DecimalType(10,6)))
    .withColumn("fiftytwoweekhigh", spark_max("dm_high").over(w_52).cast(DecimalType(8,2)))
    .withColumn("fiftytwoweeklow", spark_min("dm_low").over(w_52).cast(DecimalType(8,2)))
    .select(
        col("sk_securityid"),
        col("sk_companyid"),
        col("sk_dateid"),
        col("peratio"),
        col("yield"),
        col("fiftytwoweekhigh"),
        col("fiftytwoweeklow"),
        col("dm_close").cast(DecimalType(8,2)).alias("closeprice"),
        col("dm_high").cast(DecimalType(8,2)).alias("dayhigh"),
        col("dm_low").cast(DecimalType(8,2)).alias("daylow"),
        col("dm_vol").cast("bigint").alias("volume"),
        lit(batch_id).alias("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts"),
        col("dm_date")
    )
    .drop("dm_date")
)

print(f"Joined rows : {df_gold_market.count():,}")

In [0]:
df_gold_market.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"{gold_db}.fact_markethistory")

In [0]:
EXPECTED_COUNTS = {
    "1": 5_270_304,
    "2": 5_277_664,
    "3": 5_285_024
}
recon_results = []

market_count = spark.table(f"{gold_db}.fact_markethistory").count()
expected     = EXPECTED_COUNTS[batch_id]
status       = "PASS" if market_count == expected else "FAIL"
source_count = silver_count

print(f"Source rows  : {source_count:,}")
print(f"Gold rows    : {market_count:,}")
print(f"Expected     : {expected:,}")
print(f"Status       : {status}")

recon_results.append(Row(
        source_table = "fact_markethistory",
        batch_id     = f"Batch{batch_id}",
        source_count = source_count,
        target_count = market_count,
        status       = status
    ))


In [0]:
%run ../../02_common_utils/operations

In [0]:
recon_df = spark.createDataFrame(recon_results)

for row in recon_results:
    if "ERROR" not in row.status:
        log_pipeline_recon(
            spark        = spark,
            run_id       = run_id,
            batch_id     = row.batch_id,
            domain       = "MARKET",
            table_name   = row.source_table,
            source_layer = "silver",
            target_layer = "gold",
            source_count = row.source_count,
            target_count = row.target_count
        )
        log_audit_event(
            spark         = spark,
            run_id        = run_id,
            batch         = row.batch_id,
            layer         = "gold",
            table_name    = row.source_table,
            operation     = "OVERWRITE",
            rows_affected = row.target_count
        )

display(recon_df)